# Project 15 — Local Contract Clause Finder

## Retrieve Risky or Important Clauses from Contracts

**Goal:** Build a local contract review assistant that indexes contract clauses
with risk metadata, retrieves relevant clauses for legal questions, generates
risk assessments, and produces structured risk reports — all running locally.

**Stack:** Ollama · LangChain · ChromaDB · Pydantic · Jupyter

```
Contract clauses ──► Documents (with risk metadata)
                                │
                      Ollama Embeddings ──► ChromaDB
                                                │
Legal question ──► retrieval ──► relevant clauses ──► LLM ──► Risk Analysis
                                                               │
                                           Structured output ──► Risk Report
```

### What You'll Learn

1. Index contract clauses with risk-level metadata
2. Build a risk-aware retrieval and analysis chain
3. Use Pydantic structured output for risk reports
4. Filter clauses by risk level using metadata
5. Generate actionable contract review recommendations

### Prerequisites

- **Ollama** running with `nomic-embed-text` and `qwen3:8b`
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
!pip install -q langchain langchain-ollama langchain-community langchain-text-splitters chromadb pydantic

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests

OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running — {len(models)} model(s) available")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE}: {e}")

## Step 2 — Configure LLM and Embeddings

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

LLM_MODEL = "qwen3:8b"
EMBED_MODEL = "nomic-embed-text"

llm = ChatOllama(model=LLM_MODEL, temperature=0)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

vec = embeddings.embed_query("test")
print(f"✅ Embedding model ready — dimension: {len(vec)}")
resp = llm.invoke("Say hello.")
print(f"✅ LLM ready — response: {resp.content[:80]}")

## Step 3 — Create Sample Contract Clauses

We create 8 realistic contract clauses, each tagged with:
- `clause_type` — the category (liability, termination, IP, etc.)
- `section` — the section number in the contract
- `risk` — risk level (low / medium / high)

Risk metadata enables filtered retrieval: "show me all high-risk clauses."

In [ ]:
from langchain_core.documents import Document

contract_clauses = [
    Document(
        page_content=(
            "LIMITATION OF LIABILITY: In no event shall either party be liable for any "
            "indirect, incidental, special, consequential, or punitive damages, regardless "
            "of the cause of action. The total cumulative liability of Provider shall not "
            "exceed the fees paid by Customer in the 12 months preceding the claim. This "
            "limitation applies even if Provider has been advised of the possibility of "
            "such damages."
        ),
        metadata={"clause_type": "liability", "section": "8.1", "risk": "high"},
    ),
    Document(
        page_content=(
            "INDEMNIFICATION: Provider shall indemnify, defend, and hold harmless Customer "
            "from any third-party claims arising from Provider's gross negligence or willful "
            "misconduct. Customer shall indemnify Provider from claims arising from Customer's "
            "misuse of the services or violation of applicable law."
        ),
        metadata={"clause_type": "indemnification", "section": "8.2", "risk": "medium"},
    ),
    Document(
        page_content=(
            "TERMINATION FOR CONVENIENCE: Either party may terminate this agreement for "
            "any reason with 30 days written notice. Upon termination, Customer shall pay "
            "all fees for services rendered through the termination date. Provider shall "
            "provide reasonable transition assistance for up to 60 days after termination."
        ),
        metadata={"clause_type": "termination", "section": "9.2", "risk": "medium"},
    ),
    Document(
        page_content=(
            "NON-COMPETE AND NON-SOLICITATION: During the term and for 24 months after "
            "termination, neither party shall directly solicit employees of the other party. "
            "This restriction applies globally with no geographic limitation. Violation of "
            "this clause entitles the non-breaching party to injunctive relief and liquidated "
            "damages of $100,000 per violation."
        ),
        metadata={"clause_type": "non_compete", "section": "11.1", "risk": "high"},
    ),
    Document(
        page_content=(
            "DATA PROCESSING: Provider shall process personal data only as instructed by "
            "Customer and in compliance with applicable data protection laws (GDPR, CCPA). "
            "Provider implements AES-256 encryption at rest and TLS 1.3 in transit. Data "
            "breach notification to Customer within 72 hours of discovery. Provider shall "
            "not transfer data outside the agreed jurisdiction without prior written consent."
        ),
        metadata={"clause_type": "data_protection", "section": "7.3", "risk": "medium"},
    ),
    Document(
        page_content=(
            "INTELLECTUAL PROPERTY: All IP created during the engagement belongs exclusively "
            "to Customer (work-for-hire). Provider retains rights to pre-existing IP and "
            "general knowledge/skills. Provider grants Customer a perpetual, irrevocable, "
            "royalty-free license to all pre-existing IP incorporated into deliverables. "
            "Provider warrants that deliverables do not infringe third-party IP rights."
        ),
        metadata={"clause_type": "intellectual_property", "section": "6.1", "risk": "high"},
    ),
    Document(
        page_content=(
            "PAYMENT TERMS: Invoices are due within 30 days of receipt (Net 30). Late "
            "payments accrue interest at 1.5% per month or the maximum legal rate, whichever "
            "is lower. Customer may dispute invoices in good faith within 15 days of receipt "
            "by providing written notice with specific reasons for the dispute."
        ),
        metadata={"clause_type": "payment", "section": "4.2", "risk": "low"},
    ),
    Document(
        page_content=(
            "FORCE MAJEURE: Neither party shall be liable for delays or failure to perform "
            "due to events beyond reasonable control, including natural disasters, acts of "
            "war, pandemics, government orders, or widespread infrastructure failures. The "
            "affected party must notify the other within 5 business days. If force majeure "
            "continues for more than 90 days, either party may terminate without liability."
        ),
        metadata={"clause_type": "force_majeure", "section": "12.1", "risk": "low"},
    ),
]

print(f"✅ Created {len(contract_clauses)} contract clauses:")
for c in contract_clauses:
    m = c.metadata
    print(f"   Section {m['section']:5s} | {m['clause_type']:22s} | risk={m['risk']:6s} | {len(c.page_content)} chars")

## Step 4 — Build the Contract Index

Index all clauses in ChromaDB with their risk metadata preserved.

In [ ]:
from langchain_community.vectorstores import Chroma
from pathlib import Path
import shutil

CHROMA_DIR = "sample_data/contract_chroma"
Path("sample_data").mkdir(exist_ok=True)

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    contract_clauses, embeddings,
    persist_directory=CHROMA_DIR, collection_name="contracts",
)
print(f"✅ Contract index ready — {vectorstore._collection.count()} clauses indexed")

## Step 5 — Test Retrieval by Topic and by Risk Level

We test two retrieval strategies:
1. **Semantic search** — find clauses relevant to a topic
2. **Metadata filter** — find all high-risk clauses regardless of query

In [ ]:
# Semantic search
print("=== Semantic Search: 'liability limitations' ===")
results = vectorstore.similarity_search_with_score("liability limitations", k=3)
for i, (doc, score) in enumerate(results):
    m = doc.metadata
    print(f"   [{i+1}] score={score:.4f} | {m['clause_type']:20s} | risk={m['risk']} | {doc.page_content[:60]}...")

# Metadata-filtered search: all high-risk clauses
print("\n=== Metadata Filter: all high-risk clauses ===")
high_risk = vectorstore.similarity_search(
    "contract terms", k=10, filter={"risk": "high"}
)
for doc in high_risk:
    m = doc.metadata
    print(f"   Section {m['section']} | {m['clause_type']:20s} | {doc.page_content[:70]}...")

## Step 6 — Build the Contract Review Chain

The prompt instructs the LLM to:
1. Quote key language from the clause
2. Assess the risk level
3. Flag concerning terms
4. Suggest modifications to protect the client

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

system_prompt = (
    "You are a contract review assistant. Analyze the contract clauses provided "
    "and answer the question.\n\n"
    "For each relevant clause:\n"
    "1. Quote the key language\n"
    "2. Assess risk level (low/medium/high) and explain why\n"
    "3. Flag any concerning terms or one-sided provisions\n"
    "4. Suggest specific modifications to better protect the client\n\n"
    "Contract Clauses:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def clause_query(question: str) -> dict:
    """Query the contract review assistant."""
    docs = retriever.invoke(question)
    context_text = "\n\n---\n\n".join(
        f"[Section {d.metadata['section']} — {d.metadata['clause_type']} (risk: {d.metadata['risk']})]\n{d.page_content}"
        for d in docs
    )
    response = (prompt | llm | StrOutputParser()).invoke(
        {"context": context_text, "input": question}
    )
    return {"answer": response, "sources": docs}


print("✅ Contract review chain ready!")

## Step 7 — Test Contract Review Questions

Let's ask questions a legal reviewer would typically ask.

In [ ]:
queries = [
    "What are the liability limitations in this contract?",
    "Is there a non-compete clause and what are its terms?",
    "How is intellectual property handled?",
    "What are the data protection obligations?",
    "What happens if a force majeure event occurs?",
    "What are the payment terms and late payment penalties?",
]

for q in queries:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    result = clause_query(q)
    print(f"\nA: {result['answer']}")
    sections = [s.metadata['section'] for s in result['sources']]
    print(f"\n⚖️ Sections reviewed: {sections}")

## Step 8 — Structured Risk Report with Pydantic

We use `llm.with_structured_output()` to generate a structured risk report.
Pydantic models define the output schema, and the LLM fills in the fields.

This is useful for automated contract screening workflows.

In [ ]:
from pydantic import BaseModel, Field
from typing import List


class ClauseRisk(BaseModel):
    """Risk assessment for a single contract clause."""
    clause_type: str = Field(description="Type of clause (e.g., liability, IP, termination)")
    section: str = Field(description="Section number in the contract")
    risk_level: str = Field(description="low, medium, or high")
    concern: str = Field(description="Primary concern with this clause")
    recommendation: str = Field(description="Suggested modification")


class ContractRiskReport(BaseModel):
    """Overall contract risk assessment report."""
    overall_risk: str = Field(description="Overall risk: low, medium, high, or critical")
    high_risk_clauses: List[ClauseRisk] = Field(description="List of high-risk clauses")
    recommendations: List[str] = Field(description="Top recommendations for the client")


# Generate structured report
risk_llm = llm.with_structured_output(ContractRiskReport)

all_text = "\n\n".join(
    f"[Section {c.metadata['section']} — {c.metadata['clause_type']} (risk: {c.metadata['risk']})]\n{c.page_content}"
    for c in contract_clauses
)

report = risk_llm.invoke(
    f"Generate a comprehensive risk report for these contract clauses. "
    f"Focus on high-risk items:\n\n{all_text}"
)

print(f"⚖️ OVERALL RISK: {report.overall_risk.upper()}")
print(f"\nHigh-Risk Clauses ({len(report.high_risk_clauses)}):")
for c in report.high_risk_clauses:
    print(f"  Section {c.section} ({c.clause_type}) — Risk: {c.risk_level}")
    print(f"    Concern: {c.concern}")
    print(f"    → {c.recommendation}")
print(f"\nTop Recommendations:")
for i, r in enumerate(report.recommendations, 1):
    print(f"  {i}. {r}")

## Step 9 — Interactive Contract Helper

In [ ]:
def review(question: str) -> str:
    """Ask the contract review assistant."""
    result = clause_query(question)
    print(f"Answer:\n{result['answer']}\n")
    print(f"Clauses reviewed ({len(result['sources'])}):")
    for i, doc in enumerate(result['sources']):
        m = doc.metadata
        print(f"   [{i+1}] Section {m['section']} ({m['clause_type']}, risk={m['risk']})")
    return result['answer']

_ = review("What are the most concerning clauses in this contract?")

## Limitations & Tradeoffs

| Aspect | What happens | How to improve |
|--------|-------------|----------------|
| **Simulated clauses** | We created sample clauses | Use real contract PDFs with clause extraction |
| **Legal accuracy** | LLM is not a lawyer | Always have human legal review |
| **Clause segmentation** | We manually separated clauses | Use NLP-based clause boundary detection |
| **Jurisdiction** | Generic terms | Add jurisdiction-specific analysis |
| **Structured output** | May fail on complex structures | Add retry logic and validation |

### Important Disclaimer
This tool is for educational purposes only. It does NOT provide legal advice.
Always consult qualified legal professionals for real contract reviews.

## What You Learned

1. **Risk-tagged indexing** — storing clauses with risk-level metadata
2. **Metadata-filtered search** — retrieving all high-risk clauses directly
3. **Legal analysis prompting** — instructing the LLM to quote, assess, and suggest
4. **Pydantic structured output** — generating typed risk reports from the LLM
5. **Contract screening workflow** — combining retrieval with structured analysis

## Exercises

1. **Add more clauses** — add warranty, SLA, and audit rights clauses
2. **Compare contracts** — index two contracts and compare their terms
3. **Red flag detection** — build a prompt that specifically flags one-sided terms
4. **Clause similarity** — find which clauses across contracts are most similar
5. **Export report** — save the structured risk report as JSON and markdown

---

*Next project: **16 — Local Course Tutor***